# APTOS 2019 Blindness Detection: EDA

??????????? ?????? ?????? ??? ?????? ????????????? ????????????? ??????????? ?? 5 ???????.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project directory:", PROJECT_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from config import TRAIN_CSV, IMAGE_DIR, CLASS_NAMES, NUM_CLASSES, RANDOM_SEED
from dataset import get_class_weights
from utils import set_seed

set_seed(RANDOM_SEED)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nClasses:", sorted(df["diagnosis"].unique()))

In [ ]:
class_counts = df["diagnosis"].value_counts().sort_index()
class_table = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "count": [class_counts.get(i, 0) for i in range(NUM_CLASSES)],
})
class_table["percent"] = class_table["count"] / len(df) * 100
class_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(class_table["class_name"], class_table["count"], color="#3b82f6")
ax.set_title("Class distribution")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Image count")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
weights = get_class_weights()
weights_df = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "weight": weights.numpy(),
})
weights_df

In [ ]:
sample_paths = list(IMAGE_DIR.glob("*.png"))[:20]
image_info = []
for path in sample_paths:
    with Image.open(path) as image:
        image_info.append({"image": path.name, "width": image.width, "height": image.height, "mode": image.mode})

pd.DataFrame(image_info).head(20)

In [ ]:
samples = []
for class_id in range(NUM_CLASSES):
    class_df = df[df["diagnosis"] == class_id]
    if not class_df.empty:
        samples.append(class_df.sample(1, random_state=RANDOM_SEED).iloc[0])

fig, axes = plt.subplots(1, len(samples), figsize=(18, 4))
if len(samples) == 1:
    axes = [axes]

for ax, row in zip(axes, samples):
    image_path = IMAGE_DIR / f"{row['id_code']}.png"
    image = Image.open(image_path).convert("RGB")
    ax.imshow(image)
    ax.set_title(f"{row['diagnosis']} - {CLASS_NAMES[int(row['diagnosis'])]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
missing_images = []
for image_id in df["id_code"]:
    if not (IMAGE_DIR / f"{image_id}.png").exists():
        missing_images.append(image_id)

print("Missing images:", len(missing_images))
missing_images[:10]